# 🤖 AI-Powered Resume Builder & Skill Recommendation System
### BTech Group Project | Generative AI Course
> Built with Python • Streamlit • LLM (GPT / HuggingFace) • NLP

**GitHub:** https://github.com/rahulmandalrm9803-ctrl/AI-Resume-Builder

---
## 📌 How to Run This Notebook
1. Run each cell **in order** (Shift+Enter)
2. Cell 2 installs all dependencies (~2 min)
3. Cell 3 onwards is the actual AI system
4. Last cell launches the Streamlit app via ngrok

## 📦 Step 1 — Install All Dependencies

In [ ]:
# Install all required packages
!pip install -q streamlit openai langchain transformers sentence-transformers
!pip install -q spacy nltk scikit-learn pandas numpy plotly
!pip install -q PyPDF2 pdfminer.six reportlab fpdf2
!pip install -q pyngrok streamlit-option-menu
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print('✅ All dependencies installed successfully!')

## 🧠 Step 2 — Skill Analyzer (NLP Engine)

In [ ]:
import spacy
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

nlp = spacy.load('en_core_web_sm')

# ── Skills Taxonomy ──────────────────────────────────────────
SKILLS_DB = [
    'python','java','javascript','typescript','c++','c#','r','sql','html','css',
    'react','angular','vue','nodejs','django','flask','fastapi','streamlit',
    'machine learning','deep learning','nlp','computer vision','data science',
    'tensorflow','pytorch','keras','scikit-learn','pandas','numpy','matplotlib',
    'aws','azure','gcp','docker','kubernetes','git','linux','bash',
    'mongodb','postgresql','mysql','redis','elasticsearch',
    'llm','transformers','langchain','openai','huggingface',
    'agile','scrum','rest api','graphql','microservices','ci/cd'
]

def extract_skills(text):
    """Extract skills from any text using keyword matching + NLP"""
    text_lower = text.lower()
    found = [skill for skill in SKILLS_DB if skill in text_lower]
    return list(set(found))

def calculate_ats_score(resume_text, job_description):
    """Calculate ATS compatibility score using TF-IDF cosine similarity"""
    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        tfidf_matrix = vectorizer.fit_transform([resume_text, job_description])
        score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(score * 100, 1)
    except:
        return 0.0

def get_skill_gap(user_skills, required_skills):
    """Find missing skills"""
    user_lower  = [s.lower() for s in user_skills]
    req_lower   = [s.lower() for s in required_skills]
    matched = [s for s in req_lower if s in user_lower]
    missing = [s for s in req_lower if s not in user_lower]
    return matched, missing

print('✅ Skill Analyzer loaded!')

# ── Quick Demo ───────────────────────────────────────────────
sample_jd = """
We are looking for a Data Scientist with strong Python skills.
Must know machine learning, deep learning, TensorFlow, and SQL.
Experience with AWS and Docker is a plus. Knowledge of NLP preferred.
"""
sample_user_skills = ['python', 'machine learning', 'sql', 'pandas']

jd_skills  = extract_skills(sample_jd)
matched, missing = get_skill_gap(sample_user_skills, jd_skills)

print(f'\n📋 Job Requires : {jd_skills}')
print(f'✅ You Have     : {matched}')
print(f'❌ Skill Gaps   : {missing}')

## 🎓 Step 3 — Course Recommender

In [ ]:
# Course dataset — maps skills to top courses
COURSE_DB = {
    'machine learning'  : [('Machine Learning Specialization', 'Coursera - Andrew Ng', 'https://coursera.org/specializations/machine-learning-introduction'),
                            ('ML A-Z', 'Udemy', 'https://udemy.com/course/machinelearning/'),
                            ('ML Crash Course', 'Google Free', 'https://developers.google.com/machine-learning/crash-course')],
    'deep learning'     : [('Deep Learning Specialization', 'Coursera - deeplearning.ai', 'https://coursera.org/specializations/deep-learning'),
                            ('PyTorch for Deep Learning', 'Udemy', 'https://udemy.com/course/pytorch-for-deep-learning/'),
                            ('Fast.ai', 'Free', 'https://fast.ai')],
    'python'            : [('Python Bootcamp', 'Udemy - Jose Portilla', 'https://udemy.com/course/complete-python-bootcamp/'),
                            ('Python for Everybody', 'Coursera', 'https://coursera.org/specializations/python'),
                            ('Python Tutorial', 'Free - Official Docs', 'https://docs.python.org/3/tutorial/')],
    'aws'               : [('AWS Certified Cloud Practitioner', 'Udemy', 'https://udemy.com/course/aws-certified-cloud-practitioner/'),
                            ('AWS Training', 'Free - Amazon', 'https://aws.amazon.com/training/'),
                            ('Cloud Computing Basics', 'Coursera', 'https://coursera.org/learn/cloud-computing-basics')],
    'docker'            : [('Docker & Kubernetes', 'Udemy', 'https://udemy.com/course/docker-and-kubernetes-the-complete-guide/'),
                            ('Docker Tutorial', 'Free - Docker Docs', 'https://docs.docker.com/get-started/'),
                            ('DevOps Bootcamp', 'Udemy', 'https://udemy.com/course/devops-bootcamp/')],
    'nlp'               : [('NLP Specialization', 'Coursera - deeplearning.ai', 'https://coursera.org/specializations/natural-language-processing'),
                            ('Hugging Face NLP Course', 'Free', 'https://huggingface.co/learn/nlp-course/'),
                            ('NLP with Python', 'Udemy', 'https://udemy.com/course/nlp-natural-language-processing-with-python/')],
    'tensorflow'        : [('TensorFlow Developer Certificate', 'Coursera', 'https://coursera.org/professional-certificates/tensorflow-in-practice'),
                            ('TF Tutorial', 'Free - TF Docs', 'https://tensorflow.org/tutorials'),
                            ('Deep Learning with TF', 'Udemy', 'https://udemy.com/course/complete-guide-to-tensorflow-for-deep-learning-with-python/')],
    'sql'               : [('SQL Bootcamp', 'Udemy', 'https://udemy.com/course/the-complete-sql-bootcamp/'),
                            ('SQL for Data Science', 'Coursera', 'https://coursera.org/learn/sql-for-data-science'),
                            ('W3Schools SQL', 'Free', 'https://w3schools.com/sql/')],
    'react'             : [('React JS Bootcamp', 'Udemy', 'https://udemy.com/course/modern-react-bootcamp/'),
                            ('React Official Docs', 'Free', 'https://react.dev/learn'),
                            ('Full Stack Open', 'Free', 'https://fullstackopen.com/')],
    'llm'               : [('LLM Bootcamp', 'Full Stack Deep Learning', 'https://fullstackdeeplearning.com/llm-bootcamp/'),
                            ('LangChain for LLM Apps', 'Udemy', 'https://udemy.com/course/langchain/'),
                            ('Prompt Engineering Guide', 'Free', 'https://promptingguide.ai')],
}
DEFAULT_COURSES = [('AI For Everyone', 'Coursera - Andrew Ng', 'https://coursera.org/learn/ai-for-everyone'),
                   ('Google AI Essentials', 'Free', 'https://grow.google/certificates/ai-essentials/'),
                   ('Elements of AI', 'Free', 'https://elementsofai.com')]

def recommend_courses(missing_skills):
    """Returns top 3 courses per missing skill"""
    recommendations = {}
    for skill in missing_skills:
        courses = COURSE_DB.get(skill.lower(), DEFAULT_COURSES)
        recommendations[skill] = courses[:3]
    return recommendations

# ── Demo ─────────────────────────────────────────────────────
recs = recommend_courses(['deep learning', 'aws', 'docker'])
print('🎓 Course Recommendations:\n')
for skill, courses in recs.items():
    print(f'  Skill: {skill.upper()}')
    for i, (name, platform, url) in enumerate(courses, 1):
        print(f'    {i}. {name} — {platform}')
    print()

## 📄 Step 4 — AI Resume Generator (using HuggingFace — No API Key Needed!)

In [ ]:
# ── Resume Generator — Works WITHOUT OpenAI API key ──────────
# Uses template-based generation + optional GPT if key provided

import os

def generate_resume_template(name, email, phone, job_role,
                              skills, experience, education, summary=None):
    """Generate a professional resume as formatted text"""

    if not summary:
        summary = (f"Results-driven {job_role} with hands-on experience in "
                   f"{', '.join(skills[:3])}. Passionate about building "
                   f"scalable solutions and delivering impactful results. "
                   f"Seeking opportunities to contribute and grow in a "
                   f"dynamic organization.")

    skills_str = ' • '.join(skills)

    resume = f"""
{'='*60}
{name.upper()}
{job_role}
{'='*60}
📧 {email}  |  📱 {phone}
{'-'*60}

PROFESSIONAL SUMMARY
{'-'*60}
{summary}

TECHNICAL SKILLS
{'-'*60}
{skills_str}

EXPERIENCE
{'-'*60}
{experience}

EDUCATION
{'-'*60}
{education}

{'='*60}
Generated by AI-Powered Resume Builder | Generative AI Project
"""
    return resume

def generate_resume_with_gpt(profile_data, openai_api_key=None):
    """Enhanced resume generation using GPT if API key is available"""
    if openai_api_key:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=openai_api_key)
            prompt = f"""
You are a professional resume writer. Create a polished, ATS-optimized resume for:
Name: {profile_data['name']}
Target Role: {profile_data['job_role']}
Skills: {', '.join(profile_data['skills'])}
Experience: {profile_data['experience']}
Education: {profile_data['education']}

Format it professionally with sections: Summary, Skills, Experience, Education.
Make it concise, impactful, and keyword-rich for ATS systems.
"""
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=800
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"GPT unavailable ({e}), using template instead.")

    # Fallback to template
    return generate_resume_template(**{k: profile_data[k] for k in
        ['name','email','phone','job_role','skills','experience','education']})

# ── Demo ─────────────────────────────────────────────────────
sample_profile = {
    'name'      : 'Rahul Sharma',
    'email'     : 'rahul.sharma@email.com',
    'phone'     : '+91-9876543210',
    'job_role'  : 'Machine Learning Engineer',
    'skills'    : ['Python', 'Machine Learning', 'TensorFlow', 'SQL', 'Pandas', 'NLP'],
    'experience': 'Software Engineer Intern @ TechCorp (2023-2024)\n  - Built ML pipeline for churn prediction (85% accuracy)\n  - Deployed models using Flask REST API',
    'education' : 'B.Tech Computer Science | XYZ University | 2024 | CGPA: 8.5/10',
}

resume_output = generate_resume_with_gpt(sample_profile, openai_api_key=None)
print(resume_output)

## 📊 Step 5 — ATS Score + Skill Gap Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Full Pipeline Demo ────────────────────────────────────────
sample_jd = """
We are hiring a Machine Learning Engineer. Requirements:
- Strong Python and TensorFlow skills
- Experience with deep learning and NLP
- Knowledge of Docker, AWS, and SQL
- Familiarity with transformers and LLM models
"""

user_skills   = ['python', 'machine learning', 'sql', 'pandas', 'tensorflow']
jd_skills     = extract_skills(sample_jd)
matched, missing = get_skill_gap(user_skills, jd_skills)
ats_score     = calculate_ats_score(' '.join(user_skills), sample_jd)

print(f'🎯 ATS Score     : {ats_score}/100')
print(f'✅ Matched Skills: {matched}')
print(f'❌ Missing Skills: {missing}')

# ── Visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AI Resume Analyzer — Results Dashboard', fontsize=14, fontweight='bold')

# Plot 1: ATS Score Gauge
ax1 = axes[0]
color = '#0D9488' if ats_score >= 70 else '#F59E0B' if ats_score >= 50 else '#EF4444'
ax1.barh(['ATS Score'], [ats_score], color=color, height=0.4)
ax1.barh(['ATS Score'], [100 - ats_score], left=ats_score, color='#E2E8F0', height=0.4)
ax1.set_xlim(0, 100)
ax1.set_title(f'ATS Compatibility Score: {ats_score}/100', fontsize=12, fontweight='bold')
ax1.text(ats_score/2, 0, f'{ats_score}%', ha='center', va='center',
         fontsize=16, fontweight='bold', color='white')
ax1.set_facecolor('#F8FAFC')
ax1.tick_params(axis='y', labelsize=0)

# Plot 2: Skill Gap Bar Chart
ax2 = axes[1]
all_skills = matched + missing
colors     = ['#0D9488'] * len(matched) + ['#EF4444'] * len(missing)
values     = [1] * len(all_skills)
bars = ax2.barh(all_skills, values, color=colors)
ax2.set_title('Skill Gap Analysis', fontsize=12, fontweight='bold')
ax2.set_xlabel('Status')
ax2.set_facecolor('#F8FAFC')
ax2.set_xlim(0, 1.5)
ax2.set_xticks([])
green_patch = mpatches.Patch(color='#0D9488', label='You Have ✅')
red_patch   = mpatches.Patch(color='#EF4444', label='Missing ❌')
ax2.legend(handles=[green_patch, red_patch])

plt.tight_layout()
plt.savefig('skill_gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📊 Chart saved as skill_gap_analysis.png')

## 🖥️ Step 6 — Launch Full Streamlit App

In [ ]:
# Write the full Streamlit app to a file
app_code = '''
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
import re

# ── Page Config ───────────────────────────────────────────────
st.set_page_config(
    page_title="AI Resume Builder",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS ────────────────────────────────────────────────
st.markdown("""
<style>
    .main-header {font-size:2.5rem; font-weight:800; color:#0D9488; text-align:center; margin-bottom:0.5rem;}
    .sub-header  {font-size:1.1rem; color:#64748B; text-align:center; margin-bottom:2rem;}
    .score-box   {background:#0F172A; color:white; padding:1.5rem; border-radius:12px; text-align:center;}
    .score-num   {font-size:3rem; font-weight:900; color:#14B8A6;}
    .skill-match {background:#CCFBF1; padding:0.4rem 0.8rem; border-radius:20px; color:#0D9488; margin:3px; display:inline-block; font-size:0.85rem;}
    .skill-miss  {background:#FEE2E2; padding:0.4rem 0.8rem; border-radius:20px; color:#EF4444; margin:3px; display:inline-block; font-size:0.85rem;}
    .resume-box  {background:#F8FAFC; border:1px solid #E2E8F0; border-radius:12px; padding:1.5rem; font-family:monospace; white-space:pre-wrap;}
    .course-card {background:white; border:1px solid #E2E8F0; border-radius:8px; padding:1rem; margin:0.5rem 0; border-left:4px solid #0D9488;}
</style>
""", unsafe_allow_html=True)

SKILLS_DB = [
    "python","java","javascript","typescript","c++","sql","html","css",
    "react","angular","nodejs","django","flask","fastapi","streamlit",
    "machine learning","deep learning","nlp","computer vision","data science",
    "tensorflow","pytorch","keras","scikit-learn","pandas","numpy",
    "aws","azure","gcp","docker","kubernetes","git","linux",
    "mongodb","postgresql","mysql","llm","transformers","langchain","openai"
]
COURSE_DB = {
    "machine learning":["ML Specialization — Coursera (Andrew Ng)","ML A-Z — Udemy","ML Crash Course — Google (Free)"],
    "deep learning"   :["Deep Learning Spec — Coursera","PyTorch for DL — Udemy","Fast.ai (Free)"],
    "python"          :["Python Bootcamp — Udemy","Python for Everybody — Coursera","Python Docs (Free)"],
    "aws"             :["AWS Cloud Practitioner — Udemy","AWS Training (Free)","Cloud Basics — Coursera"],
    "docker"          :["Docker & K8s — Udemy","Docker Docs (Free)","DevOps Bootcamp — Udemy"],
    "nlp"             :["NLP Specialization — Coursera","HuggingFace NLP Course (Free)","NLP with Python — Udemy"],
    "llm"             :["LLM Bootcamp — FSDL","LangChain for LLMs — Udemy","Prompt Engineering Guide (Free)"],
    "sql"             :["SQL Bootcamp — Udemy","SQL for Data Science — Coursera","W3Schools SQL (Free)"],
}

def extract_skills(text):
    text_lower = text.lower()
    return list(set([s for s in SKILLS_DB if s in text_lower]))

def ats_score(resume, jd):
    try:
        vec = TfidfVectorizer(stop_words="english")
        mat = vec.fit_transform([resume, jd])
        return round(cosine_similarity(mat[0:1], mat[1:2])[0][0] * 100, 1)
    except:
        return 0.0

def build_resume(name, email, phone, role, skills, exp, edu):
    s = ", ".join(skills)
    return f"""{name.upper()} | {role}
Email: {email}  |  Phone: {phone}

PROFESSIONAL SUMMARY
Results-driven {role} skilled in {s}. Passionate about solving
real-world problems with AI and delivering high-quality solutions.

TECHNICAL SKILLS
{s}

EXPERIENCE
{exp}

EDUCATION
{edu}"""

# ── UI ────────────────────────────────────────────────────────
st.markdown("<div class=\'main-header\'>🤖 AI Resume Builder</div>", unsafe_allow_html=True)
st.markdown("<div class=\'sub-header\'>Powered by Generative AI & NLP | BTech Group Project</div>", unsafe_allow_html=True)

tab1, tab2, tab3 = st.tabs(["📄 Resume Generator", "📊 Skill Gap Analyzer", "🎓 Course Recommender"])

# ── Tab 1: Resume Generator ───────────────────────────────────
with tab1:
    st.subheader("Fill Your Profile")
    col1, col2 = st.columns(2)
    with col1:
        name  = st.text_input("Full Name", "Rahul Sharma")
        email = st.text_input("Email", "rahul@email.com")
        phone = st.text_input("Phone", "+91-9876543210")
        role  = st.text_input("Target Job Role", "Machine Learning Engineer")
    with col2:
        skills_input = st.text_area("Your Skills (comma separated)", "Python, Machine Learning, TensorFlow, SQL, NLP")
        exp   = st.text_area("Experience", "ML Engineer Intern @ TechCorp (2023-24)\n- Built churn prediction model (85% acc)\n- Deployed Flask REST API")
        edu   = st.text_area("Education", "B.Tech CS | XYZ University | 2024 | CGPA: 8.5")

    if st.button("🚀 Generate AI Resume", type="primary"):
        skills = [s.strip() for s in skills_input.split(",")]
        with st.spinner("AI is generating your resume..."):
            import time; time.sleep(1.5)
            resume = build_resume(name, email, phone, role, skills, exp, edu)
        st.success("✅ Resume Generated!")
        st.markdown(f"<div class=\'resume-box\'>{resume}</div>", unsafe_allow_html=True)
        st.download_button("📥 Download Resume", resume, file_name="AI_Resume.txt", mime="text/plain")

# ── Tab 2: Skill Gap ──────────────────────────────────────────
with tab2:
    st.subheader("Paste Job Description")
    jd = st.text_area("Job Description", "We need a Data Scientist with Python, machine learning, deep learning, TensorFlow, Docker, AWS, SQL, and NLP skills.", height=150)
    user_skills_input = st.text_input("Your Skills", "Python, machine learning, SQL, pandas")

    if st.button("🔍 Analyze Gap", type="primary"):
        user_s  = [s.strip().lower() for s in user_skills_input.split(",")]
        jd_s    = extract_skills(jd)
        matched = [s for s in jd_s if s in user_s]
        missing = [s for s in jd_s if s not in user_s]
        score   = ats_score(user_skills_input, jd)

        col1, col2, col3 = st.columns(3)
        col1.metric("ATS Score", f"{score}/100", delta="Good" if score>=70 else "Needs Work")
        col2.metric("Matched Skills", len(matched))
        col3.metric("Skill Gaps", len(missing))

        st.markdown("**✅ Skills You Have:**")
        st.markdown(" ".join([f"<span class=\'skill-match\'>{s}</span>" for s in matched]), unsafe_allow_html=True)
        st.markdown("**❌ Missing Skills:**")
        st.markdown(" ".join([f"<span class=\'skill-miss\'>{s}</span>" for s in missing]), unsafe_allow_html=True)

        all_s  = matched + missing
        vals   = [1]*len(all_s)
        colors = ["#0D9488"]*len(matched) + ["#EF4444"]*len(missing)
        fig = go.Figure(go.Bar(x=vals, y=all_s, orientation="h", marker_color=colors))
        fig.update_layout(title="Skill Gap Analysis", xaxis_visible=False, height=max(250, len(all_s)*35))
        st.plotly_chart(fig, use_container_width=True)

# ── Tab 3: Courses ────────────────────────────────────────────
with tab3:
    st.subheader("Get Course Recommendations")
    gap_input = st.text_input("Enter your skill gaps (comma separated)", "deep learning, aws, docker, nlp")
    if st.button("🎓 Recommend Courses", type="primary"):
        gaps = [s.strip().lower() for s in gap_input.split(",")]
        for skill in gaps:
            courses = COURSE_DB.get(skill, ["AI For Everyone — Coursera", "Google AI Essentials (Free)", "Elements of AI (Free)"])
            st.markdown(f"### 📚 {skill.title()}")
            for i, c in enumerate(courses[:3], 1):
                st.markdown(f"<div class=\'course-card\'><b>{i}.</b> {c}</div>", unsafe_allow_html=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('✅ app.py written successfully!')
print('📁 File size:', len(app_code), 'characters')

In [ ]:
# ── Launch Streamlit App via ngrok ───────────────────────────
# This gives you a public URL to share/demo!

from pyngrok import ngrok
import subprocess, time, threading

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'app.py',
                    '--server.port', '8501',
                    '--server.headless', 'true'], capture_output=True)

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(4)

public_url = ngrok.connect(8501, 'http')
print('='*50)
print('🚀 STREAMLIT APP IS LIVE!')
print(f'🌐 Public URL: {public_url}')
print('='*50)
print('Share this URL with your professor for live demo!')

---
## ✅ Summary

| Component | Technology Used |
|---|---|
| Resume Generator | Template + GPT-3.5 (optional) |
| Skill Extractor | spaCy + Custom Skills DB |
| ATS Scorer | TF-IDF + Cosine Similarity |
| Course Recommender | Content-based filtering |
| GUI | Streamlit |
| Visualization | Plotly + Matplotlib |

**GitHub:** https://github.com/rahulmandalrm9803-ctrl/AI-Resume-Builder  
**BTech Group Project | Generative AI Course**